# Phase 3 — BEV (Bird's Eye View)
## 카메라 모델 + IPM 직접 구현 + YOLOv8 연동

### 학습 목표
- **핀홀 카메라 모델** — K(내부), R/t(외부) 파라미터 수학 이해
- **투영 변환** — 3D 월드 → 2D 이미지 직접 구현
- **IPM (Inverse Perspective Mapping)** — Homography로 BEV 변환 직접 구현
- **YOLOv8 + IPM** — Detection bbox → BEV 지면 좌표 변환
- **거리 추정** — BEV에서 픽셀 → 실제 미터 변환

### Phase 1~2 → Phase 3 연결
```
Detection (Phase 1)  → BEV에서 객체 위치 표시
Tracking  (Phase 2)  → BEV에서 궤적 추적
Pose      (Phase 2)  → BEV에서 heading angle 활용
```


## 0. 환경 확인


In [ ]:
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False
from pathlib import Path

print(f'NumPy : {np.__version__}')
print(f'OpenCV: {cv2.__version__}')


## 1. 핀홀 카메라 모델

### 왜 카메라 모델이 필요한가?
BEV 변환의 본질은 **3D 세계를 2D 이미지로 투영한 것을 되돌리는 것**.  
이를 위해 카메라가 어떻게 3D → 2D 투영을 수행했는지 알아야 합니다.

### 투영 과정 (4단계)
```
3D 월드 좌표 (Xw, Yw, Zw)
       ↓  [외부 파라미터: R, t]
3D 카메라 좌표 (Xc, Yc, Zc)
       ↓  [내부 파라미터: K]
2D 정규화 좌표 (x, y)
       ↓  [픽셀 변환]
2D 픽셀 좌표 (u, v)
```

### 내부 파라미터 행렬 K
$$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$

| 파라미터 | 의미 |
|----------|------|
| $f_x, f_y$ | 초점 거리 (픽셀 단위) |
| $c_x, c_y$ | 주점 (이미지 중심) |

### 외부 파라미터 [R | t]
- **R**: 카메라 회전 (pitch = 앞으로 기울기, yaw = 좌우 회전, roll = 기울기)
- **t**: 카메라 위치 (자율주행에서 height가 핵심)


In [ ]:
def build_camera_matrix(fx, fy, cx, cy):
    """내부 파라미터 행렬 K"""
    return np.array([
        [fx,  0, cx],
        [ 0, fy, cy],
        [ 0,  0,  1]
    ], dtype=np.float64)


def build_rotation_matrix(pitch_deg, yaw_deg=0, roll_deg=0):
    """
    카메라 회전 행렬.
    pitch: 카메라가 아래를 향하는 각도 (자율주행 차량에서 ~15~20도)
    """
    pitch = np.radians(pitch_deg)
    yaw   = np.radians(yaw_deg)
    roll  = np.radians(roll_deg)

    Rx = np.array([  # pitch (x축 회전)
        [1,           0,            0],
        [0,  np.cos(pitch), -np.sin(pitch)],
        [0,  np.sin(pitch),  np.cos(pitch)],
    ])
    Ry = np.array([  # yaw (y축 회전)
        [ np.cos(yaw), 0, np.sin(yaw)],
        [0,            1,           0],
        [-np.sin(yaw), 0, np.cos(yaw)],
    ])
    Rz = np.array([  # roll (z축 회전)
        [np.cos(roll), -np.sin(roll), 0],
        [np.sin(roll),  np.cos(roll), 0],
        [0,             0,            1],
    ])
    return Rz @ Ry @ Rx


def project_3d_to_2d(points_3d, K, R, t):
    """
    3D 월드 좌표 → 2D 픽셀 좌표 투영.
    points_3d: (N, 3)  [X, Y, Z] in world frame
    반환: (N, 2) pixel coordinates
    """
    # 월드 → 카메라 좌표
    pts_cam = (R @ points_3d.T + t.reshape(3, 1)).T  # (N, 3)

    # 카메라 → 픽셀
    pts_h = (K @ pts_cam.T).T  # (N, 3) homogeneous
    pts_2d = pts_h[:, :2] / pts_h[:, 2:3]  # normalize
    return pts_2d


# ── 카메라 파라미터 설정 (자율주행 차량 기준) ──
IMG_W, IMG_H = 1280, 720
CAM_HEIGHT   = 1.5   # 카메라 지면으로부터 높이 (미터)
PITCH_DEG    = 15.0  # 카메라 아래 기울기 (도)

K = build_camera_matrix(
    fx=700, fy=700,
    cx=IMG_W/2, cy=IMG_H/2
)
R = build_rotation_matrix(pitch_deg=PITCH_DEG)
t = np.array([0, CAM_HEIGHT, 0], dtype=np.float64)  # 카메라 위치

print('카메라 내부 파라미터 K:')
print(K)
print()
print(f'카메라 높이: {CAM_HEIGHT}m,  Pitch: {PITCH_DEG}도')

# 지면(Z=0) 위의 3D 점들이 어디로 투영되는지 확인
test_pts = np.array([
    [0,   0, 5],   # 정면 5m
    [0,   0, 10],  # 정면 10m
    [0,   0, 20],  # 정면 20m
    [2,   0, 10],  # 오른쪽 2m, 정면 10m
    [-2,  0, 10],  # 왼쪽 2m, 정면 10m
], dtype=np.float64)

projected = project_3d_to_2d(test_pts, K, R, t)
print()
print('3D 지면 좌표 → 2D 픽셀 투영')
print(f'{"3D (X,Y,Z)":20s} → {"2D (u,v)"}')
for pt3, pt2 in zip(test_pts, projected):
    print(f'  {str(tuple(pt3)):20s} → ({pt2[0]:.1f}, {pt2[1]:.1f})')


## 2. IPM (Inverse Perspective Mapping)

### 핵심 아이디어
투영 변환의 역 — **지면(Z=0) 위의 점**이라는 가정 하에  
이미지 픽셀을 실제 지면 좌표로 역변환

### Homography 행렬
지면(Z=0)으로 가정하면 3D → 2D 투영이 **3×3 Homography H**로 단순화:

$$\begin{bmatrix}u\\v\\1\end{bmatrix} \sim H \begin{bmatrix}X_w\\Y_w\\1\end{bmatrix}$$

**IPM = H의 역변환**: 이미지 픽셀 (u,v) → 지면 좌표 (Xw, Yw)

### 구현 방법
```
방법 1: 수학적 유도 (K, R, t → H 계산)
방법 2: 4점 대응 (cv2.getPerspectiveTransform)
         → 이미지의 4점 ↔ BEV의 4점 지정
         → 실용적, CARLA/실차에서 캘리브레이션으로 구함
```


In [ ]:
def compute_ipm_homography(K, R, t, cam_height):
    """
    카메라 파라미터로 IPM Homography 계산.
    지면(Y=0, 자동차 좌표계) 평면 가정.
    """
    # 지면 법선벡터 n = [0, 1, 0] (y축이 위 방향)
    # 지면까지 거리 d = cam_height
    # H = K * (R - t*nT/d) 의 앞 두 열 + 세 번째 열 구성

    n = np.array([0, 1, 0], dtype=np.float64)
    d = cam_height

    # 지면 호모그래피: H_gnd = R - (t * nT) / d
    H_gnd = R - np.outer(t, n) / d

    # 3×3 으로 축소 (X, Z 좌표만 사용, Y=0)
    H = K @ H_gnd[:, [0, 2, 1]]  # X, Z, (Y=0 상수항)
    return H


def four_point_ipm(src_pts, bev_size=(600, 400), bev_range_m=(20, 10)):
    """
    4점 대응으로 IPM Homography 계산 (실용적 방법).

    src_pts: 이미지에서의 4점 (사다리꼴 ROI)
             [좌하, 우하, 우상, 좌상] 순서
    bev_size: BEV 이미지 크기 (w, h)
    bev_range_m: BEV가 커버하는 실제 거리 (전방 m, 좌우 m)

    반환: H (3×3), dst_pts
    """
    bev_w, bev_h = bev_size
    # BEV에서의 4점 (직사각형)
    dst_pts = np.float32([
        [0,     bev_h],   # 좌하 (가까운 쪽)
        [bev_w, bev_h],   # 우하
        [bev_w, 0    ],   # 우상 (먼 쪽)
        [0,     0    ],   # 좌상
    ])
    src_pts = np.float32(src_pts)
    H = cv2.getPerspectiveTransform(src_pts, dst_pts)
    return H, dst_pts


print('IPM Homography 함수 정의 완료')
H_math = compute_ipm_homography(K, R, t, CAM_HEIGHT)
print(f'수학적 Homography H shape: {H_math.shape}')


## 3. 합성 도로 이미지 생성 + IPM 적용

실제 드라이빙 영상 없이도 IPM을 검증할 수 있도록  
차선, 차량, 보행자가 있는 합성 도로 이미지를 생성합니다.


In [ ]:
def create_synthetic_road(width=1280, height=720):
    """
    합성 도로 이미지 생성:
    - 회색 도로 + 흰색 차선
    - 색깔 박스로 차량/보행자 표시
    """
    img = np.ones((height, width, 3), dtype=np.uint8) * 50  # 어두운 배경

    # 하늘 (파랑)
    img[:height//2, :] = [135, 180, 220]

    # 소실점 (이미지 중앙 위)
    vp_x, vp_y = width // 2, height // 2 - 30

    # 도로 영역 (사다리꼴)
    road_pts = np.array([
        [0,          height],
        [width,      height],
        [width*2//3, height//2],
        [width//3,   height//2],
    ], dtype=np.int32)
    cv2.fillPoly(img, [road_pts], (100, 100, 100))  # 회색 도로

    # 차선 (흰색 점선) — 소실점으로 수렴
    lane_offsets = [-0.3, 0, 0.3]  # 차선 위치 비율
    for off in lane_offsets:
        for y_frac in np.arange(0.55, 1.0, 0.08):
            y1 = int(height * y_frac)
            y2 = int(height * (y_frac + 0.04))
            t_top = (y1 - vp_y) / (height - vp_y)
            t_bot = (y2 - vp_y) / (height - vp_y)
            x_top = int(vp_x + off * width * 0.45 * t_top)
            x_bot = int(vp_x + off * width * 0.45 * t_bot)
            w_top = max(2, int(4 * t_top))
            w_bot = max(2, int(4 * t_bot))
            pts = np.array([[x_top-w_top, y1],[x_top+w_top, y1],
                            [x_bot+w_bot, y2],[x_bot-w_bot, y2]])
            cv2.fillPoly(img, [pts], (255, 255, 255))

    # 차량 (파란 박스, 원근감 적용)
    vehicles = [
        (0.5,  0.65, 0.08, 0.10, (200, 80, 80)),   # 정면 가까운 차
        (0.35, 0.60, 0.06, 0.07, (80, 80, 200)),   # 왼쪽 차
        (0.65, 0.58, 0.05, 0.06, (80, 200, 80)),   # 오른쪽 차
        (0.50, 0.56, 0.04, 0.04, (200, 80, 200)),  # 멀리
    ]
    obj_positions = []  # (이미지 bbox, 실제 라벨)
    for cx_r, cy_r, w_r, h_r, color in vehicles:
        cx = int(cx_r * width)
        cy = int(cy_r * height)
        w  = int(w_r * width)
        h  = int(h_r * height)
        x1, y1 = cx - w//2, cy - h//2
        x2, y2 = cx + w//2, cy + h//2
        cv2.rectangle(img, (x1, y1), (x2, y2), color, -1)
        cv2.rectangle(img, (x1, y1), (x2, y2), (255,255,255), 1)
        obj_positions.append((x1, y1, x2, y2, 'car'))

    # 보행자 (작은 박스)
    pedestrians = [
        (0.25, 0.62, 0.02, 0.06, (255, 200, 100)),
        (0.75, 0.60, 0.02, 0.05, (255, 200, 100)),
    ]
    for cx_r, cy_r, w_r, h_r, color in pedestrians:
        cx = int(cx_r * width)
        cy = int(cy_r * height)
        w  = int(w_r * width)
        h  = int(h_r * height)
        x1, y1 = cx - w//2, cy - h//2
        x2, y2 = cx + w//2, cy + h//2
        cv2.rectangle(img, (x1, y1), (x2, y2), color, -1)
        obj_positions.append((x1, y1, x2, y2, 'person'))

    return img, obj_positions


road_img, obj_positions = create_synthetic_road(IMG_W, IMG_H)
print(f'합성 도로 이미지 생성: {road_img.shape}')
print(f'객체 수: {len(obj_positions)}')
for obj in obj_positions:
    print(f'  {obj[4]:6s}: ({obj[0]},{obj[1]}) ~ ({obj[2]},{obj[3]})')


In [ ]:
# ── ROI 4점 설정 + IPM 적용 ──

# 이미지에서 도로 ROI 4점 (사다리꼴)
# [좌하, 우하, 우상, 좌상] — 소실점으로 수렴하는 사다리꼴
BEV_W, BEV_H = 600, 500

src_pts = [
    [IMG_W * 0.05,  IMG_H * 0.98],   # 좌하
    [IMG_W * 0.95,  IMG_H * 0.98],   # 우하
    [IMG_W * 0.62,  IMG_H * 0.52],   # 우상 (소실점 근처)
    [IMG_W * 0.38,  IMG_H * 0.52],   # 좌상
]

H_ipm, dst_pts = four_point_ipm(src_pts, bev_size=(BEV_W, BEV_H))

# IPM 적용
bev_img = cv2.warpPerspective(road_img, H_ipm, (BEV_W, BEV_H))

# ── 시각화 ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 원본 이미지 + ROI 표시
vis_orig = road_img.copy()
roi_pts  = np.array(src_pts, dtype=np.int32)
cv2.polylines(vis_orig, [roi_pts], True, (0, 255, 255), 3)
for pt in roi_pts:
    cv2.circle(vis_orig, tuple(pt), 8, (0, 255, 0), -1)

axes[0].imshow(cv2.cvtColor(vis_orig, cv2.COLOR_BGR2RGB))
axes[0].set_title('원본 카메라 이미지 (청록색: ROI 4점)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(bev_img, cv2.COLOR_BGR2RGB))
axes[1].set_title('IPM 변환 결과 — Bird\'s Eye View', fontsize=12)
axes[1].set_xlabel('← 왼쪽 / 오른쪽 →')
axes[1].set_ylabel('↑ 먼 곳 / 가까운 곳 ↓')

plt.suptitle('Inverse Perspective Mapping (IPM)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('ipm_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('차선이 BEV에서 평행선이 되면 IPM 성공')


## 4. BEV 픽셀 → 실제 거리 변환

BEV의 1픽셀이 실제로 몇 미터인지 알면 **거리 추정** 가능.

```
BEV 이미지 높이 = BEV_H 픽셀 = 전방 RANGE_M 미터
→ 스케일: meters_per_pixel = RANGE_M / BEV_H
```


In [ ]:
# 실제 거리 매핑 설정
RANGE_FORWARD_M = 30.0  # BEV가 커버하는 전방 거리 (미터)
RANGE_LATERAL_M = 15.0  # BEV가 커버하는 좌우 거리 (미터)

m_per_px_y = RANGE_FORWARD_M / BEV_H   # 세로 방향
m_per_px_x = RANGE_LATERAL_M / BEV_W   # 가로 방향

print(f'BEV 스케일: {m_per_px_y:.4f} m/px (전방), {m_per_px_x:.4f} m/px (측면)')
print()


def pixel_to_meters(px, py, bev_h=BEV_H, bev_w=BEV_W,
                    range_fwd=RANGE_FORWARD_M, range_lat=RANGE_LATERAL_M):
    """
    BEV 픽셀 (px, py) → 실제 거리 (forward_m, lateral_m)
    py=0: 먼 쪽, py=bev_h: 가까운 쪽
    px=0: 왼쪽, px=bev_w: 오른쪽
    """
    forward_m = (bev_h - py) * (range_fwd / bev_h)
    lateral_m = (px - bev_w / 2) * (range_lat / bev_w)
    return forward_m, lateral_m


def img_bbox_to_bev(bbox_xyxy, H_ipm):
    """
    이미지 bbox의 하단 중앙점 → BEV 좌표 변환.
    하단 중앙 = 객체가 지면에 닿는 점 (IPM 가정: 지면 위의 점)
    """
    x1, y1, x2, y2 = bbox_xyxy
    foot_x = (x1 + x2) / 2
    foot_y = y2  # bbox 하단

    pt = np.array([[[foot_x, foot_y]]], dtype=np.float32)
    bev_pt = cv2.perspectiveTransform(pt, H_ipm)[0][0]
    return bev_pt[0], bev_pt[1]  # bev_x, bev_y


# 객체 위치 변환
print('이미지 객체 → BEV 좌표 → 실제 거리')
print('-' * 55)
for (x1, y1, x2, y2, label) in obj_positions:
    bev_x, bev_y = img_bbox_to_bev((x1, y1, x2, y2), H_ipm)
    fwd, lat = pixel_to_meters(bev_x, bev_y)
    if 0 <= bev_x <= BEV_W and 0 <= bev_y <= BEV_H:
        side = '오른쪽' if lat > 0 else '왼쪽'
        print(f'  {label:6s}: 전방 {fwd:5.1f}m, {side} {abs(lat):.1f}m')
    else:
        print(f'  {label:6s}: BEV 범위 밖')


In [ ]:
# ── BEV 위에 Detection 결과 표시 ──
bev_vis = bev_img.copy()

label_colors = {'car': (80, 80, 200), 'person': (100, 200, 255)}

for (x1, y1, x2, y2, label) in obj_positions:
    bev_x, bev_y = img_bbox_to_bev((x1, y1, x2, y2), H_ipm)
    if not (0 <= bev_x <= BEV_W and 0 <= bev_y <= BEV_H):
        continue
    fwd, lat = pixel_to_meters(bev_x, bev_y)
    color = label_colors.get(label, (200, 200, 200))
    cv2.circle(bev_vis, (int(bev_x), int(bev_y)), 8, color, -1)
    cv2.putText(bev_vis, f'{label}\n{fwd:.1f}m',
                (int(bev_x)+8, int(bev_y)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

# 자차 위치 (BEV 하단 중앙)
ego_x, ego_y = BEV_W // 2, BEV_H - 20
cv2.drawMarker(bev_vis, (ego_x, ego_y), (0, 255, 255),
               cv2.MARKER_TRIANGLE_UP, 20, 2)

# 거리 눈금선
for dist_m in [5, 10, 15, 20, 25]:
    py = int(BEV_H - dist_m * BEV_H / RANGE_FORWARD_M)
    if 0 <= py <= BEV_H:
        cv2.line(bev_vis, (0, py), (BEV_W, py), (80, 80, 80), 1)
        cv2.putText(bev_vis, f'{dist_m}m', (5, py-3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (150, 150, 150), 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(cv2.cvtColor(road_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('카메라 이미지', fontsize=12)
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(bev_vis, cv2.COLOR_BGR2RGB))
axes[1].set_title('BEV — 객체 위치 + 거리 추정', fontsize=12)
axes[1].set_xlabel('← 왼쪽 / 오른쪽 →')

plt.suptitle('YOLOv8 Detection → BEV 좌표 변환 + 거리 추정', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('bev_detection.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. YOLOv8 실제 이미지 연동

합성 이미지가 아닌 COCO128 실제 이미지에 IPM 적용.  
실제 드라이빙 이미지가 없으므로 COCO 이미지로 파이프라인 검증.


In [ ]:
from ultralytics import YOLO
from pathlib import Path

project_root = Path('C:/Users/apple/Desktop/autonomous_cv_pipeline')
det_model = YOLO(project_root / 'phase1_basics/detection/runs/detect/coco128_finetune/weights/best.pt')

data_path = Path.home() / 'MyProject' / 'datasets' / 'coco128'
img_paths = sorted((data_path / 'images' / 'train2017').glob('*.jpg'))[:3]

# 이미지별 ROI는 이미지 크기에 맞게 조정
def get_road_roi(w, h):
    """이미지 크기에 맞는 기본 도로 ROI 4점"""
    return [
        [w * 0.05, h * 0.98],
        [w * 0.95, h * 0.98],
        [w * 0.65, h * 0.55],
        [w * 0.35, h * 0.55],
    ]

fig, axes = plt.subplots(len(img_paths), 3, figsize=(18, 6*len(img_paths)))
if len(img_paths) == 1:
    axes = [axes]

for row, img_path in enumerate(img_paths):
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]

    # Detection
    result = det_model(img_path, conf=0.3, verbose=False)[0]
    det_img = img.copy()
    detections = []
    if len(result.boxes):
        for box in result.boxes:
            x1,y1,x2,y2 = map(int, box.xyxy[0])
            cls_name = det_model.names[int(box.cls[0])]
            cv2.rectangle(det_img, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(det_img, cls_name, (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
            detections.append((x1,y1,x2,y2,cls_name))

    # IPM
    roi = get_road_roi(w, h)
    H_img, _ = four_point_ipm(roi, bev_size=(400, 400))
    bev = cv2.warpPerspective(img, H_img, (400, 400))

    # BEV에 detection 표시
    bev_vis2 = bev.copy()
    for (x1,y1,x2,y2,lbl) in detections:
        bx, by = img_bbox_to_bev((x1,y1,x2,y2), H_img)
        if 0 <= bx <= 400 and 0 <= by <= 400:
            cv2.circle(bev_vis2, (int(bx), int(by)), 6, (0,255,0), -1)
            cv2.putText(bev_vis2, lbl, (int(bx)+5, int(by)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0,255,0), 1)

    axes[row][0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[row][0].set_title('원본', fontsize=10)
    axes[row][0].axis('off')

    axes[row][1].imshow(cv2.cvtColor(det_img, cv2.COLOR_BGR2RGB))
    axes[row][1].set_title('Detection', fontsize=10)
    axes[row][1].axis('off')

    axes[row][2].imshow(cv2.cvtColor(bev_vis2, cv2.COLOR_BGR2RGB))
    axes[row][2].set_title('BEV + Detection', fontsize=10)
    axes[row][2].axis('off')

plt.suptitle('YOLOv8 + IPM 연동 파이프라인', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('yolo_bev_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. 멀티뷰 BEV — 개념 이해

### 단일 카메라 IPM의 한계
```
1. 지면이 평평하다는 가정 필요 (경사로, 과속방지턱 오류)
2. 가려진 영역(occlusion) 복원 불가
3. 먼 거리일수록 왜곡 심함
```

### 멀티뷰 BEV (nuScenes: 6개 카메라)
```
전방(CAM_FRONT) + 전방좌(CAM_FRONT_LEFT) + 전방우(CAM_FRONT_RIGHT)
+후방(CAM_BACK) + 후방좌(CAM_BACK_LEFT) + 후방우(CAM_BACK_RIGHT)
         ↓
각 카메라 이미지 → BEV feature 추출
         ↓
BEV feature 합성 (Spatial Transformer / Cross-attention)
         ↓
360° BEV 맵 생성
```

### 최신 논문 접근 방법
| 논문 | 방법 | 핵심 |
|------|------|------|
| BEVDet (2022) | LSS (Lift-Splat-Shoot) | 깊이 예측 → voxel lifting |
| BEVFusion (2022) | Camera + LiDAR 융합 | 멀티모달 BEV |
| BEVFormer (2022) | Transformer attention | 시간적 BEV 누적 |

**IPM은 이 모든 방법의 기초** — 좌표 변환 원리는 동일.


In [ ]:
# ── 멀티뷰 BEV 시뮬레이션 (개념 시각화) ──
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# 6개 카메라 방향 시뮬레이션
cam_configs = [
    ('전방 (FRONT)',        0,   (0.35, 0.10, 0.65, 0.50)),
    ('전방좌 (FRONT_LEFT)', -60, (0.00, 0.10, 0.40, 0.50)),
    ('전방우 (FRONT_RIGHT)', 60, (0.60, 0.10, 1.00, 0.50)),
    ('후방 (BACK)',         180, (0.35, 0.50, 0.65, 0.90)),
    ('후방좌 (BACK_LEFT)', -120, (0.00, 0.50, 0.40, 0.90)),
    ('후방우 (BACK_RIGHT)',  120, (0.60, 0.50, 1.00, 0.90)),
]

road_rgb = cv2.cvtColor(road_img, cv2.COLOR_BGR2RGB)
bev_combined = np.zeros((BEV_H, BEV_W, 3), dtype=np.uint8)

for i, (name, yaw, _) in enumerate(cam_configs):
    # 각 카메라 방향으로 회전된 IPM 시뮬레이션
    R_yaw = build_rotation_matrix(pitch_deg=15, yaw_deg=yaw)
    # 합성 이미지를 회전해 다른 방향 카메라처럼 보이게
    M = cv2.getRotationMatrix2D((IMG_W//2, IMG_H//2), -yaw*0.3, 1.0)
    rotated = cv2.warpAffine(road_img, M, (IMG_W, IMG_H))

    H_cam, _ = four_point_ipm(src_pts, bev_size=(BEV_W, BEV_H))
    bev_cam  = cv2.warpPerspective(rotated, H_cam, (BEV_W, BEV_H))

    ax = axes[i // 3][i % 3]
    ax.imshow(cv2.cvtColor(bev_cam, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{name}\nBEV', fontsize=9)
    ax.axis('off')

    # 합성 (단순 평균)
    bev_combined = cv2.addWeighted(bev_combined, 0.7, bev_cam, 0.3, 0)

# 통합 BEV
ax_combined = axes[1][3]
# 자차 위치 표시
bev_comb_vis = bev_combined.copy()
cv2.drawMarker(bev_comb_vis, (BEV_W//2, BEV_H//2),
               (0, 255, 255), cv2.MARKER_STAR, 25, 3)
ax_combined.imshow(cv2.cvtColor(bev_comb_vis, cv2.COLOR_BGR2RGB))
ax_combined.set_title('통합 BEV (6-cam 융합)', fontsize=9, fontweight='bold')
ax_combined.axis('off')

# 남은 칸
axes[0][3].axis('off')

plt.suptitle('멀티뷰 BEV — 6개 카메라 → 360° BEV 통합 개념', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('multiview_bev.png', dpi=150, bbox_inches='tight')
plt.show()
print('실제 BEVFusion/BEVDet은 Transformer로 feature 수준에서 융합')


## 8. 최종 결과 요약 및 Phase 4 연결


In [ ]:
print('=' * 60)
print('Phase 3 BEV 파이프라인 최종 결과')
print('=' * 60)
print()
print('[구현 완료]')
print('  1. 핀홀 카메라 모델  — K, R, t 수식 직접 구현')
print('  2. 3D → 2D 투영     — project_3d_to_2d 직접 구현')
print('  3. IPM              — 4점 Homography + warpPerspective')
print('  4. 픽셀 → 미터       — BEV 스케일 변환')
print('  5. YOLOv8 연동      — Detection bbox → BEV 좌표')
print('  6. 멀티뷰 BEV       — 6-camera 통합 개념 시각화')
print()
print('[거리 추정 결과]')
for (x1, y1, x2, y2, label) in obj_positions:
    bev_x, bev_y = img_bbox_to_bev((x1, y1, x2, y2), H_ipm)
    if 0 <= bev_x <= BEV_W and 0 <= bev_y <= BEV_H:
        fwd, lat = pixel_to_meters(bev_x, bev_y)
        side = '우' if lat > 0 else '좌'
        print(f'  {label:6s}: 전방 {fwd:5.1f}m, {side} {abs(lat):.1f}m')
print()
print('[Phase 1~3 통합 파이프라인]')
print('  카메라 이미지')
print('  → YOLOv8 Detection (Phase 1)')
print('  → ByteTrack Tracking (Phase 2)')
print('  → IPM BEV 변환 (Phase 3)')
print('  → 객체별 전방/측면 거리 추정')
print()
print('[Phase 4 CARLA 연결]')
print('  CARLA 시뮬레이터에서 카메라 K/R/t 파라미터 직접 추출 가능')
print('  → 완벽한 GT와 함께 IPM 정확도 정량 평가 가능')
print('  → Phase 1~3 모든 모듈을 CARLA 파이프라인에 통합')
